### 1. Loading and Inspecting the Data

The shared path helper locates local `data/raw/` inputs or Kaggle competition data. Run this notebook inside the repository checkout, with its shared package installed or accessible through the setup cell. JSONL has one battle object per line.


In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

# This notebook may run from the repo root or any notebooks/<name>/ folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is not None and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pokemon_challenge.paths import data_dir, output_dir

DATA_DIR = data_dir()
OUTPUT_DIR = output_dir("aimilia", "logreg-72feat")  # logistic regression, 72 features
print(f"Using data from: {DATA_DIR}")

train_file_path = DATA_DIR / "train.jsonl"
test_file_path = DATA_DIR / "test.jsonl"
sample_submission_path = DATA_DIR / "sample_submission.csv"


def load_jsonl(path):
    """Read a .jsonl file into a list of dicts, one per line."""
    with open(path) as f:
        return [json.loads(line) for line in f]


train_data = load_jsonl(train_file_path)
test_data = load_jsonl(test_file_path)
print(f"Loaded {len(train_data)} train and {len(test_data)} test battles.")

Using data from: /Users/emilykokkinofta/Projects/FDS_Challenge_26/data
Loaded 10000 train and 5000 test battles.


In [2]:
from features_static import build_static 
from features_timeline import build_timeline
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# same feature pipeline for both sets
train = build_static(train_data, labelled=True).merge(
        build_timeline(train_data), on="battle_id")
test  = build_static(test_data, labelled=False).merge(
        build_timeline(test_data), on="battle_id")

features = [c for c in train.columns if c not in ("battle_id", "player_won")]
X_train, y_train = train[features], train["player_won"]
X_test = test[features]          # indexing by `features` fixes column order

# pipelines prevent leakage 
model = make_pipeline(
    StandardScaler(), 
    LogisticRegression(max_iter=2000, random_state=42),
)

# know your score BEFORE you submit
cv = cross_val_score(model, X_train, y_train, cv=5)
print(f"CV accuracy: {cv.mean():.4f} (+/- {cv.std():.4f})")

model.fit(X_train, y_train)
test_predictions = model.predict(X_test)

CV accuracy: 0.6419 (+/- 0.0047)


## features

### from build_static (features before the fight starts):

* p1:
    - six pokemons.
        - 2 summaries over stat: mean and max (6*2=12)
    - 2 levels (mean, min)
    - shape (types: how many elements the team covers, dual_type: hoe many creatures have two types instead of one, special_item: how many carry something other than the defaut item) -3

total = 17 features

* p2:
    - one summary over stat per pokemon: only one pokemon shown-no team to average (6)
    - lead pokemons extras ( level, types) -2

total = 8

* differences for the two teams (p1 summary minus p2 lead)
    - 2 for each stat :mean, max (6*2=12)

total = 12

grand total from build_static = 37 features


### from build_timeline ( things that happen in the 30 runs):
each of the 30 turns stores 4 things per side: active creatures, HP percentage of the active creature, its status condition, and which move it used.

We get each of the following twice ( one per team )
- faints: how many creatures died -2
- health: hppct_mean, hppct_final, hppct_min -6
- health over time: hppct_early_mean, hppct_late_mean, hppct_trend	health in turns 1–10, turns 21–30, and the change between -6
- status condition: status_rate, slp_rate -4
- loss of control: switches: how often they swapped creatures, n_species_seen: how many different ones appeared, no_move_rate: share of turns where no move happened (switching, fainted, asleep) -6
- move quality: mean_base_power, mean_accuracy: how hard and how reliably their moves hit, status_move_rate: share of non-damaging moves — a defensive playstyle -6
- differences: diff_hppct_mean   diff_hppct_final   diff_status_rate
diff_no_move_rate   diff_switches - 5

total:  35

GRAND TOTAL: 72 features (compared to 6 of the startup notebook)



### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [3]:
sample_submission = pd.read_csv(sample_submission_path)
expected_columns = ['battle_id', 'player_won']
if list(sample_submission.columns) != expected_columns:
    raise ValueError(f"Expected {expected_columns}, got {list(sample_submission.columns)}")

predictions_by_id = pd.Series(test_predictions, index=test["battle_id"])
submission_df = sample_submission.copy()
submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id)
if submission_df['player_won'].isna().any():
    raise ValueError("Some battle_id values have no matching prediction.")
submission_df['player_won'] = submission_df['player_won'].astype(int)

submission_df.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
(OUTPUT_DIR / "cv_score.txt").write_text(f"{cv.mean():.4f} +/- {cv.std():.4f}\n")
print(f"Saved: {OUTPUT_DIR / 'submission.csv'}")

Saved: /Users/emilykokkinofta/Projects/FDS_Challenge_26/outputs/aimilia/logreg-72feat/submission.csv
